# Archived CSV exploration

Kept as a record of the initial analysis. Use `../exploratory_rds_plotly.ipynb` for current work. The 63 date-only parsing failures below were subsequently confirmed as midnight in the RDS. Paths now point to `../data/`.

# Explore black carbon measurements
Run the cells from top to bottom using **Run All**. This notebook reads the CSV in chunks so the entire file does not need to fit in memory.

Full-file results include missing values, sensor coverage, date coverage and daily BC means. Distributions, quantiles and correlations use a reproducible random sample of up to 50,000 rows from across the entire file. The source CSV is never modified.

The file does not specify measurement units or the timestamp timezone, so labels retain source units and timestamps. Negative BC values are flagged but retained. Confirm the instrument's quality-control rules before removing measurements.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# If imports fail, run this in a new cell: %pip install pandas numpy matplotlib
REPO_DIR = next((p for p in [Path.cwd(), Path.cwd().parent, Path.cwd() / "black_carbon"]
                     if (p / "data").is_dir() and (p / "README.md").is_file()), Path.cwd())
DATA_DIR = REPO_DIR / "data"
CSV_PATH = DATA_DIR / "Black Carbon Concentrations.csv"
CHUNK_SIZE = 100_000
SAMPLE_SIZE = 50_000
RANDOM_SEED = 42

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True})
files = sorted(DATA_DIR.rglob("*.csv"))
display(pd.DataFrame([{"file": str(p.relative_to(DATA_DIR)),
                       "size_MB": round(p.stat().st_size / 1_000_000, 2)} for p in files]))
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Update DATA_DIR or CSV_PATH. File not found: {CSV_PATH}")

## Preview the source
This preview shows the first ten rows only. The analysis below scans the complete file.

In [ ]:
preview = pd.read_csv(CSV_PATH, nrows=10)
display(preview)
display(preview.dtypes.rename("preview_dtype").to_frame())

## Scan the complete file
`NA` and other standard pandas missing-value markers are read as missing. Numeric parsing failures and nonfinite values are counted separately and excluded from numeric summaries. Unparseable dates are excluded from daily summaries. No rows are dropped from total row counts.

Each row gets a random priority; keeping the 50,000 smallest priorities gives a uniform sample across all chunks. Daily means are calculated from sums and valid counts, rather than averaging chunk means.

In [ ]:
NUMERIC_COLUMNS = ["Temperature", "RH", "Sense", "Reference", "Attenuation",
                   "BC", "Smoothed", "Latitude", "Longitude"]
rng = np.random.default_rng(RANDOM_SEED)
row_count = 0
missing = pd.Series(0, index=preview.columns, dtype="int64")
invalid_numeric = pd.Series(0, index=NUMERIC_COLUMNS, dtype="int64")
nonfinite = invalid_numeric.copy()
invalid_dates = negative_bc = 0
sensor_parts, daily_parts = [], []
sample = pd.DataFrame()

for chunk_number, chunk in enumerate(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE,
                                                dtype="string"), start=1):
    row_count += len(chunk)
    missing += chunk.isna().sum()
    raw_date = chunk["date"]
    chunk["date"] = pd.to_datetime(raw_date, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    invalid_dates += int((raw_date.notna() & chunk["date"].isna()).sum())
    for col in NUMERIC_COLUMNS:
        raw = chunk[col]
        values = pd.to_numeric(raw, errors="coerce").astype(float)
        invalid_numeric[col] += int((raw.notna() & values.isna()).sum())
        nonfinite[col] += int(np.isinf(values).sum())
        chunk[col] = values.replace([np.inf, -np.inf], np.nan)
    negative_bc += int(chunk["BC"].lt(0).sum())

    sensor_parts.append(chunk.groupby("SensorID", dropna=False).agg(
        rows=("BC", "size"), valid_BC=("BC", "count"),
        BC_sum=("BC", "sum"), first_date=("date", "min"), last_date=("date", "max")))
    daily_parts.append(chunk.assign(day=chunk["date"].dt.floor("D")).groupby(
        ["SensorID", "day"], dropna=False).agg(
        rows=("BC", "size"), valid_BC=("BC", "count"), BC_sum=("BC", "sum")))

    chunk["_priority"] = rng.random(len(chunk))
    sample = pd.concat([sample, chunk], ignore_index=True).nsmallest(SAMPLE_SIZE, "_priority")
    if chunk_number % 10 == 0:
        print(f"Scanned {row_count:,} rows...")

if row_count == 0:
    raise ValueError("The CSV has headers but no data rows.")
sample = sample.drop(columns="_priority").reset_index(drop=True)
sensors = pd.concat(sensor_parts).groupby(level=0, dropna=False).agg(
    rows=("rows", "sum"), valid_BC=("valid_BC", "sum"), BC_sum=("BC_sum", "sum"),
    first_date=("first_date", "min"), last_date=("last_date", "max"))
sensors["mean_BC"] = sensors.pop("BC_sum") / sensors["valid_BC"].replace(0, np.nan)
sensors = sensors.sort_values("rows", ascending=False)
daily = pd.concat(daily_parts).groupby(level=[0, 1], dropna=False).sum().reset_index()
daily = daily.loc[daily["day"].notna()].copy()
daily["mean_BC"] = daily.pop("BC_sum") / daily["valid_BC"].replace(0, np.nan)
quality = pd.DataFrame({"missing": missing, "missing_pct": missing / row_count * 100,
                        "invalid_numeric": invalid_numeric, "nonfinite_numeric": nonfinite})
quality[["invalid_numeric", "nonfinite_numeric"]] = quality[["invalid_numeric", "nonfinite_numeric"]].fillna(0).astype(int)
print(f"Finished: {row_count:,} rows; {len(sample):,} sampled rows.")

## Full-file coverage and quality

In [ ]:
display(pd.Series({
    "total_rows": row_count,
    "identified_sensors": int(sensors.index.notna().sum()),
    "first_timestamp": sensors["first_date"].min(),
    "last_timestamp": sensors["last_date"].max(),
    "unparseable_dates": invalid_dates,
    "negative_BC_rows": negative_bc,
    "negative_BC_pct_of_valid": 100 * negative_bc / sensors["valid_BC"].sum()
        if sensors["valid_BC"].sum() else np.nan,
}).to_frame("value"))
display(quality.sort_values("missing_pct", ascending=False))
display(sensors)
quality["missing_pct"].sort_values().plot.barh(title="Missing values — full file", xlabel="Percent of rows")
plt.tight_layout()
plt.show()

## Sample statistics and distributions
These quantiles describe the random sample, not exact full-file quantiles. The histogram uses a symmetric log x-axis to retain negative values and make extreme values visible.

In [ ]:
display(sample[NUMERIC_COLUMNS].describe(percentiles=[.01, .05, .5, .95, .99]).T)
bc = sample["BC"].dropna()
if not bc.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    # Transform bin edges too, so extreme values do not collapse the center into one bin.
    transformed = np.sign(bc) * np.log1p(np.abs(bc))
    edges = np.linspace(transformed.min() - 1e-9, transformed.max() + 1e-9, 81)
    bins = np.sign(edges) * np.expm1(np.abs(edges))
    axes[0].hist(bc, bins=bins)
    axes[0].set_xscale("symlog", linthresh=1)
    axes[0].set(title="Sample BC distribution", xlabel="BC (source units, symlog scale)", ylabel="Sample rows")
    ordered = np.sort(bc)
    axes[1].plot(ordered, np.arange(1, len(ordered) + 1) / len(ordered))
    axes[1].set_xscale("symlog", linthresh=1)
    axes[1].set(title="Sample cumulative distribution", xlabel="BC (source units, symlog scale)", ylabel="Fraction of sample")
    plt.tight_layout()
    plt.show()

## Daily patterns by sensor
Full-file daily means and measurement counts for the five sensors with the most rows. Means include negative BC readings and are weighted by observations, not elapsed time. Missing days appear as gaps; counts help identify uneven coverage.

In [ ]:
top_sensors = sensors.loc[sensors.index.notna()].head(5).index
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for sensor in top_sensors:
    series = daily.loc[daily["SensorID"].eq(sensor)].set_index("day").sort_index()
    if series.empty:
        continue
    series = series.reindex(pd.date_range(series.index.min(), series.index.max(), freq="D"))
    axes[0].plot(series.index, series["mean_BC"], label=sensor)
    axes[1].plot(series.index, series["valid_BC"].fillna(0), label=sensor)
axes[0].set(title="Daily mean BC — full file", ylabel="BC (source units)")
axes[1].set(title="Valid BC readings per day — full file", ylabel="Readings", xlabel="Date (source timezone unspecified)")
if axes[0].lines:
    axes[0].legend()
plt.tight_layout()
plt.show()

## Explore relationships
Sample-based Pearson correlations use pairwise complete observations. Combined-sensor relationships can reflect sensor differences or time trends and do not establish causation.

In [ ]:
correlations = sample[NUMERIC_COLUMNS].corr()
display(correlations.round(2))
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(correlations, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(NUMERIC_COLUMNS)), NUMERIC_COLUMNS, rotation=60, ha="right")
ax.set_yticks(range(len(NUMERIC_COLUMNS)), NUMERIC_COLUMNS)
ax.grid(False)
ax.set_title("Numeric correlations — random sample")
fig.colorbar(im, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

## Your own exploration
Change `SENSOR` to one of the sensor IDs above. `sample` holds sampled observations, `sensors` contains full-file sensor summaries, and `daily` contains full-file daily aggregates. Sampled observations do not support exact duplicate checks or measurement-interval analysis.

In [ ]:
SENSOR = top_sensors[0] if len(top_sensors) else None
sensor_sample = sample.loc[sample["SensorID"].eq(SENSOR)].sort_values("date")
display(sensor_sample.head(20))
# Examples to try:
# display(sample.nlargest(20, "BC"))
# display(daily.loc[daily["SensorID"].eq(SENSOR)])
# display(sample.loc[sample["BC"].lt(0)].head(20))